# Cyclone abundance-only applicability diagnostic

Inspect the full repeat-level evidence, verify exact invariance to uniform state redistribution, and reproduce the clean-reference numerical failure. This is an applicability diagnostic, separate from the four-method accuracy comparison.

Download the registered repeat-level asset before running this notebook. Environment: **chronoclone**, micromamba; `/path/to/ChronoClone/.runtime/chronoclone`; kernel **chronoclone-paper**. Versions and the MacBook execution command are displayed below. Set `JUPYTER_PATH=/path/to/ChronoClone/.runtime/jupyter` for the registered local kernel.

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import warnings

import nbclient
import nbformat
import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "results/aggregate/aggregation_metadata.json").is_file())
FROZEN = next(p / "frozen_runtime" for p in [ROOT, ROOT.parent]
              if (p / "frozen_runtime/production").is_dir())

ENVIRONMENT = {
    "manager": "micromamba (existing prefix reused without mutation)",
    "name": "chronoclone",
    "path": sys.prefix,
    "kernel": "chronoclone-paper",
    "python": sys.version.split()[0],
    "execution_command": (
        "/path/to/ChronoClone/.runtime/chronoclone/bin/python "
        "/path/to/ChronoClone/ChronoClone_paper/benchmarks/run_notebook.py /path/to/ChronoClone/ChronoClone_paper/benchmarks/ode_noise/notebooks/0004_cyclone_applicability_diagnostic.ipynb --kernel chronoclone-paper"
    ),
}
assert ENVIRONMENT["python"] == "3.11.15"
assert Path(ENVIRONMENT["path"]).is_dir()

versions = pd.DataFrame(
    {
        "component": [
            "Python", "NumPy", "pandas", "SciPy", "scikit-learn",
            "nbformat", "nbclient",
        ],
        "version": [
            ENVIRONMENT["python"], np.__version__, pd.__version__,
            scipy.__version__, sklearn.__version__, nbformat.__version__,
            nbclient.__version__,
        ],
    }
)
display(pd.DataFrame([ENVIRONMENT]))
display(versions)

,manager,name,path,kernel,python,execution_command
0,micromamba (existing prefix reused without mut...,chronoclone,/path/to/ChronoClone/.runt...,chronoclone-paper,3.11.15,/path/to/ChronoClone/.runt...


,component,version
0,Python,3.11.15
1,NumPy,2.4.6
2,pandas,2.3.3
3,SciPy,1.17.1
4,scikit-learn,1.9.0
5,nbformat,5.10.4
6,nbclient,0.11.0


## Context & Methods

### Diagnostic question

Does the Cyclone abundance-only output represent a fair accuracy comparison,
a software defect, or a method-applicability mismatch?

### Key assumptions

- The completed, hash-validated aggregate is the controlling quantitative
  source.
- `Cyclone abundance-only` denotes the published abundance/correlation/Ward
  core with fixed default K=6 and its dynamic clone filter disabled; it is not
  the complete Seurat workflow.
- Uniform state redistribution preserves every clone-time total by contract.
- A finite-fit conditional ARI is not interpreted as performance over failed
  repeats.

In [2]:
AGGREGATE = ROOT / "results" / "aggregate"
SUMMARY_PATH = AGGREGATE / "native_method_noise_summary.csv"
COMPLETENESS_PATH = AGGREGATE / "result_completeness.csv"
FAILURE_PATH = AGGREGATE / "method_failure_counts.csv"
RUNS_PATH = AGGREGATE / "native_method_noise_runs.csv.gz"
METADATA_PATH = AGGREGATE / "aggregation_metadata.json"
CONFIG_PATH = ROOT / "config" / "benchmark_config.json"

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
summary = pd.read_csv(SUMMARY_PATH)
completeness = pd.read_csv(COMPLETENESS_PATH)
failure_counts = pd.read_csv(FAILURE_PATH)

for filename, expected_hash in metadata["artifact_sha256"].items():
    assert sha256_file(AGGREGATE / filename) == expected_hash
assert metadata["status"] == "pass"
assert metadata["observed_rows"] == metadata["expected_rows"] == 1_300_500
assert metadata["cyclone_uniform_invariance_validated"] is True

display(
    pd.DataFrame(
        {
            "source": [
                SUMMARY_PATH.relative_to(ROOT),
                COMPLETENESS_PATH.relative_to(ROOT),
                FAILURE_PATH.relative_to(ROOT),
                RUNS_PATH.relative_to(ROOT),
            ],
            "grain": [
                "method x Gaussian x uniform (100-repeat summary)",
                "method x Gaussian x uniform fit coverage",
                "method x Gaussian x uniform x failure type",
                "repeat x Gaussian x uniform x method",
            ],
        }
    )
)
display(
    pd.DataFrame(
        {
            "validation": [
                "aggregate status", "terminal rows", "uniform-invariance gate",
                "Cyclone dynamic filter enabled",
            ],
            "value": [
                metadata["status"], metadata["observed_rows"],
                metadata["cyclone_uniform_invariance_validated"],
                config["cyclone"]["dynamic_clone_filter"],
            ],
        }
    )
)
large = metadata["omitted_large_artifacts"]["native_method_noise_runs.csv.gz"]
assert RUNS_PATH.stat().st_size == large["bytes"] and sha256_file(RUNS_PATH) == large["sha256"]


,source,grain
0,results/aggregate/native_method_noise_summary.csv,method x Gaussian x uniform (100-repeat summary)
1,results/aggregate/result_completeness.csv,method x Gaussian x uniform fit coverage
2,results/aggregate/method_failure_counts.csv,method x Gaussian x uniform x failure type
3,results/aggregate/native_method_noise_runs.csv.gz,repeat x Gaussian x uniform x method


,validation,value
0,aggregate status,pass
1,terminal rows,1300500
2,uniform-invariance gate,True
3,Cyclone dynamic filter enabled,False


## Data

In [3]:
method_coverage = (
    summary.groupby("method", sort=False)
    .agg(
        requested_fits=("n_expected", "sum"),
        finite_partitions=("n_success", "sum"),
        method_failures=("n_method_failure", "sum"),
        coordinates_with_any_fit=("n_success", lambda values: int((values > 0).sum())),
        total_coordinates=("n_success", "size"),
    )
    .reset_index()
)
method_coverage["finite_fit_fraction"] = (
    method_coverage["finite_partitions"] / method_coverage["requested_fits"]
)
assert method_coverage["requested_fits"].eq(260_100).all()
display(method_coverage)

cyclone_summary = summary.loc[summary["method"].eq("Cyclone abundance-only")].copy()
assert len(cyclone_summary) == 2_601
assert int(cyclone_summary["n_success"].sum()) == 11_934
assert int(cyclone_summary["n_method_failure"].sum()) == 248_166
assert failure_counts["method"].eq("Cyclone abundance-only").all()
assert failure_counts["error_type"].eq("NativeSelectionFailure").all()
assert failure_counts["error_message"].nunique() == 1
display(failure_counts[["error_type", "error_message", "count"]].groupby(
    ["error_type", "error_message"], as_index=False
).agg(failed_fits=("count", "sum")))

,method,requested_fits,finite_partitions,method_failures,coordinates_with_any_fit,total_coordinates,finite_fit_fraction
0,ChronoClone,260100,260100,0,2601,2601,1.000000
1,CLADES preprocessing,260100,260100,0,2601,2601,1.000000
2,Cyclone abundance-only,260100,11934,248166,1377,2601,0.045882
3,K-means,260100,260100,0,2601,2601,1.000000
4,Ward,260100,260100,0,2601,2601,1.000000


,error_type,error_message,failed_fits
0,NativeSelectionFailure,Cyclone R-compatible distance has a pair with ...,248166


## Results

In [4]:
cyclone_run_parts = []
run_columns = [
    "method", "repeat", "gaussian_pct", "uniform_pct", "status", "ari",
    "recovered_k", "error_type", "error_message", "method_diagnostic_json",
]
for chunk in pd.read_csv(RUNS_PATH, compression="gzip", usecols=run_columns, chunksize=100_000):
    cyclone_part = chunk.loc[chunk["method"].eq("Cyclone abundance-only")]
    if len(cyclone_part):
        cyclone_run_parts.append(cyclone_part)
cyclone_runs = pd.concat(cyclone_run_parts, ignore_index=True)
assert len(cyclone_runs) == 260_100

pair_group = cyclone_runs.groupby(["repeat", "gaussian_pct"], sort=True)
pair_size = pair_group.size()
assert pair_size.eq(51).all()

uniform_invariance_checks = pd.DataFrame(
    {
        "field": ["status", "ARI", "recovered K", "error type", "error message"],
        "maximum distinct values across 51 uniform levels": [
            int(pair_group["status"].nunique(dropna=False).max()),
            int(pair_group["ari"].nunique(dropna=False).max()),
            int(pair_group["recovered_k"].nunique(dropna=False).max()),
            int(pair_group["error_type"].nunique(dropna=False).max()),
            int(pair_group["error_message"].nunique(dropna=False).max()),
        ],
    }
)
assert uniform_invariance_checks.iloc[:, 1].eq(1).all()
display(uniform_invariance_checks)

pair_first = pair_group.first().reset_index()
successful_pairs = pair_first.loc[pair_first["status"].eq("ok")].copy()
assert len(successful_pairs) == 234
assert len(successful_pairs) * 51 == int(cyclone_summary["n_success"].sum())

by_gaussian = (
    pair_first.groupby("gaussian_pct")
    .agg(
        successful_repeats=("status", lambda values: int((values == "ok").sum())),
        requested_repeats=("status", "size"),
    )
    .reset_index()
)
by_gaussian["fit_fraction"] = (
    by_gaussian["successful_repeats"] / by_gaussian["requested_repeats"]
)

coordinate_summary = pd.DataFrame(
    {
        "quantity": [
            "requested Cyclone rows",
            "finite partitions",
            "finite-fit fraction",
            "uniform-only finite partitions",
            "independent successful repeat/Gaussian pairs",
            "Gaussian levels with at least one finite fit",
            "first Gaussian level with a finite fit",
            "maximum repeat fit fraction at any Gaussian level",
            "heatmap coordinates with any finite fit",
            "heatmap coordinates with no finite fit",
        ],
        "value": [
            len(cyclone_runs),
            int(cyclone_runs["status"].eq("ok").sum()),
            float(cyclone_runs["status"].eq("ok").mean()),
            int(cyclone_runs.loc[cyclone_runs["gaussian_pct"].eq(0), "status"].eq("ok").sum()),
            len(successful_pairs),
            int((by_gaussian["successful_repeats"] > 0).sum()),
            int(by_gaussian.loc[by_gaussian["successful_repeats"] > 0, "gaussian_pct"].min()),
            float(by_gaussian["fit_fraction"].max()),
            int(cyclone_summary["n_success"].gt(0).sum()),
            int(cyclone_summary["n_success"].eq(0).sum()),
        ],
    }
)
display(coordinate_summary)
display(by_gaussian.loc[by_gaussian["successful_repeats"].gt(0)].reset_index(drop=True))

,field,maximum distinct values across 51 uniform levels
0,status,1
1,ARI,1
2,recovered K,1
3,error type,1
4,error message,1


,quantity,value
0,requested Cyclone rows,260100.000000
1,finite partitions,11934.000000
2,finite-fit fraction,0.045882
3,uniform-only finite partitions,0.000000
4,independent successful repeat/Gaussian pairs,234.000000
5,Gaussian levels with at least one finite fit,27.000000
6,first Gaussian level with a finite fit,44.000000
7,maximum repeat fit fraction at any Gaussian level,0.220000
8,heatmap coordinates with any finite fit,1377.000000
9,heatmap coordinates with no finite fit,1224.000000


,gaussian_pct,successful_repeats,requested_repeats,fit_fraction
0,44,1,100,0.01
1,48,1,100,0.01
2,52,4,100,0.04
3,54,1,100,0.01
4,56,4,100,0.04
5,58,6,100,0.06
6,60,16,100,0.16
7,62,14,100,0.14
8,64,21,100,0.21
9,66,22,100,0.22


In [5]:
successful_pair_ari = successful_pairs["ari"].dropna()
finite_coordinate_ari = cyclone_summary.loc[
    cyclone_summary["ari_n_finite"].gt(0), "ari_mean"
]
assert len(successful_pair_ari) == len(successful_pairs)
assert successful_pairs["recovered_k"].eq(6).all()

conditional_accuracy = pd.DataFrame(
    {
        "quantity": [
            "successful independent repeat/Gaussian pairs",
            "conditional ARI mean",
            "conditional ARI median",
            "conditional ARI 2.5th percentile",
            "conditional ARI 97.5th percentile",
            "minimum coordinate-wise conditional mean ARI",
            "maximum coordinate-wise conditional mean ARI",
            "recovered K for every finite fit",
        ],
        "value": [
            len(successful_pair_ari),
            float(successful_pair_ari.mean()),
            float(successful_pair_ari.median()),
            float(successful_pair_ari.quantile(0.025)),
            float(successful_pair_ari.quantile(0.975)),
            float(finite_coordinate_ari.min()),
            float(finite_coordinate_ari.max()),
            int(successful_pairs["recovered_k"].iloc[0]),
        ],
    }
)
display(conditional_accuracy)

,quantity,value
0,successful independent repeat/Gaussian pairs,234.000000
1,conditional ARI mean,0.031480
2,conditional ARI median,0.030761
3,conditional ARI 2.5th percentile,0.020610
4,conditional ARI 97.5th percentile,0.042860
5,minimum coordinate-wise conditional mean ARI,0.023074
6,maximum coordinate-wise conditional mean ARI,0.041116
7,recovered K for every finite fit,6.000000


In [6]:
REFERENCE = ROOT / "reference"
profiles = np.loadtxt(REFERENCE / "deterministic_profiles.csv", delimiter=",").reshape(12, 4, 13)
program_manifest = pd.read_csv(REFERENCE / "program_manifest.csv")
assert len(program_manifest) == len(profiles) == 12
assert int(program_manifest["n_clones"].sum()) == 517

intended_abundance = profiles.sum(axis=2)
rounded_profiles = np.rint(profiles).astype(np.int64)
rounded_abundance = rounded_profiles.sum(axis=2)
expanded_clean_counts = np.repeat(
    rounded_profiles, program_manifest["n_clones"].to_numpy(int), axis=0
)

information_audit = pd.DataFrame(
    {
        "quantity": [
            "truth programs",
            "distinct 52-dimensional state profiles",
            "distinct intended four-time abundance trajectories",
            "distinct rounded four-time abundance trajectories",
            "eligible clone count represented by the clean tensor",
            "distinct clean t1 clone totals",
        ],
        "value": [
            len(profiles),
            len(np.unique(np.round(profiles.reshape(12, -1), 10), axis=0)),
            len(np.unique(np.round(intended_abundance, 10), axis=0)),
            len(np.unique(rounded_abundance, axis=0)),
            len(expanded_clean_counts),
            len(np.unique(expanded_clean_counts[:, 0, :].sum(axis=1))),
        ],
    }
)
display(information_audit)

program_abundance_table = program_manifest[["truth_metaclone", "n_clones"]].copy()
for time_index, time_label in enumerate(["t1", "t2", "t3", "t4"]):
    program_abundance_table[f"{time_label}_intended_total"] = intended_abundance[:, time_index]
    program_abundance_table[f"{time_label}_rounded_total"] = rounded_abundance[:, time_index]
display(program_abundance_table)

,quantity,value
0,truth programs,12
1,distinct 52-dimensional state profiles,12
2,distinct intended four-time abundance trajecto...,2
3,distinct rounded four-time abundance trajectories,8
4,eligible clone count represented by the clean ...,517
5,distinct clean t1 clone totals,1


,truth_metaclone,n_clones,t1_intended_total,t1_rounded_total,t2_intended_total,t2_rounded_total,t3_intended_total,t3_rounded_total,t4_intended_total,t4_rounded_total
0,MC01,15,10.0,10,100.0,100,300.0,301,400.0,401
1,MC02,30,10.0,10,100.0,100,300.0,301,400.0,401
2,MC03,50,10.0,10,100.0,100,300.0,303,400.0,400
3,MC04,40,10.0,10,100.0,100,300.0,299,400.0,397
4,MC05,72,10.0,10,100.0,100,300.0,301,400.0,401
5,MC06,56,10.0,10,100.0,99,300.0,299,400.0,400
6,MC07,88,10.0,10,100.0,100,300.0,300,400.0,400
7,MC08,20,10.0,10,100.0,99,300.0,298,400.0,398
8,MC09,37,10.0,10,100.0,100,0.0,0,0.0,0
9,MC10,53,10.0,10,100.0,100,300.0,300,400.0,399


In [7]:
# Relocate the frozen import adapter while retaining all of its source checks.
import importlib.util, tempfile, zipfile
adapter_path = FROZEN / "production/scripts/cyclone_adapter.py"
assert sha256_file(adapter_path) == "907525a8e9261686290e4d25829a5c1af52840d491e987cae43eefa4497a7cf6"
runtime_spec = importlib.util.spec_from_file_location("source_runtime", FROZEN / "production/scripts/source_runtime.py")
source_runtime = importlib.util.module_from_spec(runtime_spec)
runtime_spec.loader.exec_module(source_runtime)
numerical_runtime = FROZEN / "normalization_runtime"
assert sha256_file(numerical_runtime / "chronoclone/normalization.py") == "ea9626c444f3527505bd1d594e0fff2896eb34609769de108b86748f962ce9cb"
assert sha256_file(numerical_runtime / "chronoclone/__init__.py") == "1199332fe6c10e81af55eec91d583ae6da245e39bab44d86cc4112705f8c7480"
source_runtime.SOURCE_SCRIPTS = FROZEN / "native_20260803/benchmark_package/scripts"
source_runtime.SOURCE_SITE_PACKAGES = numerical_runtime
source_runtime.SOURCE_PROFILE = REFERENCE / "deterministic_profiles.csv"
source_runtime.SOURCE_PROGRAM_MANIFEST = REFERENCE / "program_manifest.csv"
source_runtime.SOURCE_MANIFEST = FROZEN / "native_20260803/results/provenance/environment_manifest.json"
sys.modules["source_runtime"] = source_runtime
sys.path.insert(0, str(FROZEN / "production/scripts"))
import cyclone_adapter

clean_abundance = expanded_clean_counts.sum(axis=2, dtype=np.int64).astype(float)
time_by_clone = (clean_abundance / clean_abundance.sum(axis=0, keepdims=True) * 100.0).T
row_means = time_by_clone.mean(axis=1, keepdims=True)
row_sd = time_by_clone.std(axis=1, ddof=1, keepdims=True)
constant_time_rows = np.ptp(clean_abundance, axis=0) == 0
row_sd[constant_time_rows] = 0.0
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    scaled = (time_by_clone - row_means) / row_sd
clean_correlation = cyclone_adapter.r_pairwise_complete_correlation(scaled)

try:
    cyclone_adapter.cyclone_correlation_core(expanded_clean_counts, n_clusters=6)
except Exception as error:
    clean_failure_type = type(error).__name__
    clean_failure_message = str(error)
else:
    raise AssertionError("The locked clean Cyclone fixture unexpectedly returned a fit")

clean_numerical_audit = pd.DataFrame(
    {
        "quantity": [
            "constant time rows after state summation",
            "finite clean clone-correlation entries",
            "all-nonfinite clean correlation profiles",
            "clean fit outcome",
            "clean failure message",
        ],
        "value": [
            int(constant_time_rows.sum()),
            int(np.isfinite(clean_correlation).sum()),
            int((~np.isfinite(clean_correlation)).all(axis=1).sum()),
            clean_failure_type,
            clean_failure_message,
        ],
    }
)
display(clean_numerical_audit)

,quantity,value
0,constant time rows after state summation,1
1,finite clean clone-correlation entries,0
2,all-nonfinite clean correlation profiles,517
3,clean fit outcome,NativeSelectionFailure
4,clean failure message,Cyclone R-compatible distance has a pair with ...


## Takeaways

In [8]:
cyclone_coverage = method_coverage.loc[
    method_coverage["method"].eq("Cyclone abundance-only")
].iloc[0]
fit_percent = 100.0 * float(cyclone_coverage["finite_fit_fraction"])
ari_mean = float(successful_pair_ari.mean())
max_success_percent = 100.0 * float(by_gaussian["fit_fraction"].max())

display(Markdown(f"""
**Observed evidence.** The abundance-only adapter returned a finite partition
for **{int(cyclone_coverage['finite_partitions']):,} of
{int(cyclone_coverage['requested_fits']):,} requested rows ({fit_percent:.2f}%)**.
Those rows reduce to only **{len(successful_pairs)} independent
repeat/Gaussian outcomes**, each copied unchanged across all 51 uniform levels.
No uniform-only condition fitted. Even where the core returned K=6, its
conditional mean ARI was **{ari_mean:.3f}**, and no Gaussian level fitted more
than **{max_success_percent:.0f}%** of repeats.

**Mechanistic diagnosis.** This is not evidence that full Cyclone is simply a
poor clustering algorithm. The adapter discards all 13-state composition and
uses only four clone-abundance totals. The 12 truth programs collapse to
{int(information_audit.loc[information_audit['quantity'].eq('distinct intended four-time abundance trajectories'), 'value'].iloc[0])}
intended abundance trajectories. Uniform redistribution is therefore exactly
invisible. The fixed clean t1 abundance row is constant, and the clean
correlation profiles are non-finite. Large Gaussian perturbations occasionally
break numerical ties by injecting random abundance variation, but that
variation is not the 12-program state-composition truth.

**Presentation decision.** The current Cyclone mean-ARI heatmap is not a fair
primary accuracy panel. It mixes gray undefined cells with conditional means
based on 1--22 successful repeats and duplicates every result down the uniform
axis. A reviewer could reasonably mistake this for a performance surface or a
full published Cyclone comparison. Cyclone should be removed from the primary
ARI figures and reported only as a clearly labelled applicability audit. A
valid Cyclone accuracy experiment would require a separate abundance-trajectory
benchmark (or suitable longitudinal biological data), the published dynamic
filter/full workflow, and a truth target defined in terms Cyclone can observe.
"""))

presentation_scope = pd.DataFrame(
    {
        "location": [
            "Primary accuracy figures",
            "Supplementary applicability audit",
            "Future Cyclone-specific benchmark",
        ],
        "Cyclone treatment": [
            "Exclude from ARI curves and heatmaps",
            "Report finite-fit fraction, exact uniform invariance, and core-only scope; do not rank by ARI",
            "Use abundance-informative truth/data and the published filter/full workflow",
        ],
    }
)
display(presentation_scope)


**Observed evidence.** The abundance-only adapter returned a finite partition
for **11,934 of
260,100 requested rows (4.59%)**.
Those rows reduce to only **234 independent
repeat/Gaussian outcomes**, each copied unchanged across all 51 uniform levels.
No uniform-only condition fitted. Even where the core returned K=6, its
conditional mean ARI was **0.031**, and no Gaussian level fitted more
than **22%** of repeats.

**Mechanistic diagnosis.** This is not evidence that full Cyclone is simply a
poor clustering algorithm. The adapter discards all 13-state composition and
uses only four clone-abundance totals. The 12 truth programs collapse to
2
intended abundance trajectories. Uniform redistribution is therefore exactly
invisible. The fixed clean t1 abundance row is constant, and the clean
correlation profiles are non-finite. Large Gaussian perturbations occasionally
break numerical ties by injecting random abundance variation, but that
variation is not the 12-program state-composition truth.

**Presentation decision.** The current Cyclone mean-ARI heatmap is not a fair
primary accuracy panel. It mixes gray undefined cells with conditional means
based on 1--22 successful repeats and duplicates every result down the uniform
axis. A reviewer could reasonably mistake this for a performance surface or a
full published Cyclone comparison. Cyclone should be removed from the primary
ARI figures and reported only as a clearly labelled applicability audit. A
valid Cyclone accuracy experiment would require a separate abundance-trajectory
benchmark (or suitable longitudinal biological data), the published dynamic
filter/full workflow, and a truth target defined in terms Cyclone can observe.


,location,Cyclone treatment
0,Primary accuracy figures,Exclude from ARI curves and heatmaps
1,Supplementary applicability audit,"Report finite-fit fraction, exact uniform inva..."
2,Future Cyclone-specific benchmark,Use abundance-informative truth/data and the p...
